In [3]:
import pandas as pd
import joblib
import numpy as np
from datetime import timedelta

# Load your pre-trained model
model = joblib.load('bus_eta_predictor.joblib')

def predict_eta(input_data, min_eta=0.1):
    """
    Predict ETA using your model with validation checks
    
    Args:
        input_data: Path to CSV or DataFrame
        min_eta: Minimum allowed ETA value in minutes
        
    Returns:
        DataFrame with predictions and validation flags
    """
    # Load data
    if isinstance(input_data, str):
        df = pd.read_csv(input_data)
    else:
        df = input_data.copy()
    
    # Convert timestamp
    df['timestamp'] = pd.to_datetime(df['timestamp'])
    
    # Get model features and ensure correct order
    model_features = model.booster_.feature_name()
    X = df[model_features]
    
    # Predict
    df['predicted_eta_minutes'] = model.predict(X)
    
    # Apply minimum ETA threshold
    df['predicted_eta_minutes'] = df['predicted_eta_minutes'].clip(lower=min_eta)
    
    # Calculate arrival time
    df['arrival_time'] = df['timestamp'] + pd.to_timedelta(df['predicted_eta_minutes'], unit='minutes')
    
    # Add validation flags
    df['is_negative_prediction'] = df['predicted_eta_minutes'] < min_eta
    df['is_low_confidence'] = df['predicted_eta_minutes'] < 1.0
    
    return df

def print_predictions(df):
    """Helper function to print predictions in readable format"""
    print("\nPredictions:")
    display_cols = ['timestamp', 'current_stop_name', 'next_stop_name', 
                   'predicted_eta_minutes', 'arrival_time',
                   'is_negative_prediction', 'is_low_confidence']
    display_cols = [c for c in display_cols if c in df.columns]
    print(df[display_cols].head().to_string(formatters={
        'timestamp': lambda x: x.strftime('%Y-%m-%d %H:%M:%S'),
        'arrival_time': lambda x: x.strftime('%Y-%m-%d %H:%M:%S'),
        'predicted_eta_minutes': '{:.2f}'.format
    }))

# Example usage
print("=== Predicting ETAs ===")

# 1. First check model features
print("\nModel expects these features:")
print(model.booster_.feature_name())

# 2. Predict with validation
print("\nPredicting for preprocessed.csv...")
preprocessed_data = predict_eta('preprocessed.csv')
print_predictions(preprocessed_data)

print("\nPredicting for bus_eta_standard_scaled.csv...")
scaled_data = predict_eta('bus_eta_standard_scaled.csv')
print_predictions(scaled_data)

# 3. Sample prediction
sample_data = pd.DataFrame({
    'timestamp': ['2024-01-01 07:00:00'],
    'stop_sequence': [2],
    'current_stop_name': [1],
    'next_stop_name': [2],
    'day_of_week': [0],
    'is_holiday': [False],
    'is_peak_hour': [False],
    'weather_condition': [1],
    'passenger_count': [20],
    'current_speed': [25.0],
    'distance_to_next_stop': [0.5],
    'current_lat': [3.05],
    'current_lon': [101.80]
})

print("\nSample prediction:")
sample_pred = predict_eta(sample_data)
print_predictions(sample_pred)

# 4. Model diagnostics
print("\n=== Model Diagnostics ===")
print("Prediction statistics:")
print(pd.Series(sample_pred['predicted_eta_minutes']).describe())

=== Predicting ETAs ===

Model expects these features:
['current_stop_name', 'next_stop_name', 'day_of_week', 'is_holiday', 'is_peak_hour', 'weather_condition', 'passenger_count', 'current_speed', 'distance_to_next_stop', 'current_lat', 'current_lon']

Predicting for preprocessed.csv...
[LightGBM] [Warning] min_data_in_leaf is set=100, min_child_samples=54 will be ignored. Current value: min_data_in_leaf=100
[LightGBM] [Warning] feature_fraction is set=0.9978199538634552, colsample_bytree=1.0 will be ignored. Current value: feature_fraction=0.9978199538634552
[LightGBM] [Warning] lambda_l1 is set=0.6280040409967319, reg_alpha=0.0 will be ignored. Current value: lambda_l1=0.6280040409967319
[LightGBM] [Warning] lambda_l2 is set=9.136391069557224, reg_lambda=0.0 will be ignored. Current value: lambda_l2=9.136391069557224
[LightGBM] [Warning] bagging_fraction is set=0.5300667902157092, subsample=1.0 will be ignored. Current value: bagging_fraction=0.5300667902157092
[LightGBM] [Warning] b